**Cell 1**

# Role 1 - Data acquisition, provenance, and raw-data EDA

**Owner: Kennedy G**

Build trustworthy raw inputs and freeze a defensible group split before anyone fits a model. You own acquisition, raw quality checks, source documentation, and raw-noise EDA, not simulation or preprocessing. Your checkpoint deliverable is data evidence, not a trained model.

Revised scope: dataset preparation, preprocessing and EDA. This is a new notebook; earlier role notebooks are preserved. Every implementation cell is deliberately empty.

**Cell 2**

## What this milestone covers

The project question remains whether four-parameter inference generalizes from O3a H1 noise to O4a H1 noise. This checkpoint concentrates on preparing and understanding the dataset: source provenance, physical labels, preprocessing, actual split counts, EDA, plots, and limitations. Model training, model performance tables and calibration conclusions are outside this notebook set's required work.

The methodology section must still describe three algorithms because the rubric asks for a baseline and two improvements: linear regression, its small regularized extension Ridge, and SNPE with a masked autoregressive flow. Describe them as the next common comparison; do not invent new training results. Feature importance is explicitly requested by the rubric. Input-target association is useful EDA but does not fully substitute for model-based importance; Role 4 must state this gap honestly if importance is deferred.

**Existing evidence versus your work:** The repository already contains saved noise, injections and processed data. The revised PDF uses those artifacts, not results from these new notebooks. Your task is a separately identified manual pilot. Do not claim your pilot has the old dataset's 800-per-run counts until you actually generate and check them. The earlier split was a 750-row O3a development pool, 50 O3a held-out examples and 800 O4a evaluation examples. A 600/150 subdivision of the development pool is a proposal, not a completed split.

**Implementation rule:** Write your own code in the empty cells. You may call standard scientific libraries and consult their documentation, but do not import ligo_pipeline, project utils, config.py, another notebook's implementation, or prepared training/model checkpoints. Trusted raw strain with source metadata is allowed. You may write your own small functions after explaining their steps. No solution code is supplied here.

**Ownership:** Assign one person per notebook; names are unassigned. Review each other's outputs, but let each owner make their own notebook changes. Record actual contributions and dates. Every cell has a sequential cell_number and every markdown cell displays its number; the preceding task labels the empty code cell. Renumber all cells from 1 when inserting new cells. Execution counters are not stable cell numbering.


**Cell 3**

## Common contract and parallel work

Agree on one small experiment before scaling up: H1; O3a development and within-run evaluation; O4a cross-run evaluation; 4096 Hz; four-second final windows; a 35-350 Hz preprocessing band; target order chirp_mass, mass_ratio, chi_eff, distance. Keep any fixed-mass, distance-only exercise separate from this four-target pilot.

Use a new experiment_id and store files under artifacts/manual_preprocessing_milestone/<experiment_id>/. The role-owned subfolders are data/, simulation/, preprocessing/ and analysis/. Put experiment.json in data/ and repeat its ID and relevant settings in each output's metadata. Store raw segment duration, sample rate, waveform assumptions, parameter ranges, seeds, intended counts, actual counts and units. Do not overwrite the existing artifacts or the older role notebooks.

Role 1 supplies raw arrays, noise_id/group_id, source GPS intervals and source split assignments. Role 3 supplies noise-only PSD files early. Role 2 uses those PSDs for SNR selection, then supplies padded injected buffers, merger indices and labels. Role 3 supplies processed arrays and row metadata. Role 4 audits and combines evidence, counts and EDA; this role does not train the final models.

**Source-group separation:** Assign groups before simulation. Overlapping raw GPS intervals are one group even if filenames differ. Use train, validation, O3a_test and O4a_test split names; keep all O4a out of fitting and tuning. A recommended small design has at least three O3a training groups, one O3a validation group, one O3a test group and two O4a groups. Choose feasible durations and target counts together; these are planning suggestions, not completed allocations. If your pilot cannot support this design, document the limitation. Track nonoverlapping final crop windows as well as group identities.

**Handoff format:** Use .npy/.npz numeric arrays and .csv/.json metadata with repo-relative paths. Join per-example records by unique sample_id; join noise/PSD records by noise_id. Preserve run, split, group and units. Never assume two files have matching row order. Save rejection ledgers, shapes, dtypes, versions and seeds. Changes to schema/settings require all consumers to check compatibility.

**Start in parallel:** Role 1 can inspect one raw segment, Role 2 can inspect clean waveforms, Role 3 can validate sinusoids/FFT/filtering, and Role 4 can design the counting ledger, plot checklist and split audit. First integrate one real example end to end through data preparation. Then expand the pilot; no model fit is needed to check these handoffs.


**Cell 4**

## Your work boundary

**Inputs:** Public GWOSC H1 O3a/O4a strain or trusted local raw .npy files with original metadata. A local downloaded file is a source, not evidence that its quality is already verified.

**Outputs:** data/experiment.json; data/noise_manifest.csv; data/noise/<noise_id>.npy; data/split_summary.csv; labeled raw-data figures and a 150-word Data/Challenges draft. Paths are under artifacts/manual_preprocessing_milestone/<experiment_id>/.

**Start in parallel:** Start with one public/local raw segment, a tiny hand-created metadata table, and descriptive plots. Freeze the table schema immediately. Deliver one vetted segment per available run early, then add split groups. Role 3 can start PSD work from the first accepted segment; Role 2 can simulate waveforms before all noise arrives.

**File contract:** noise_manifest.csv columns: noise_id, group_id, run, detector, gps_start, gps_end, sample_rate_hz, n_samples, strain_path, split, source_url_or_key, quality_status, quality_reason. gps_end is exclusive. noise_id is unique. Record all rejected candidates separately with reasons; do not delete them from the acquisition accounting.

All paths use the experiment-specific root in the common contract above. Older path wording in conceptual examples does not authorize writing into historical artifact folders.

**Cell 5**

## Task 1: Understand strain and provenance

### Background knowledge
Strain is a dimensionless time series. Sample k occurs at t0 + k/fs; N/fs is its duration, and fs/2 is the Nyquist frequency. UTC and GPS are different time standards. A long segment has millions of samples but is not millions of independent labeled examples.

### Your task
Select the pilot sources, explain UTC/GPS fields, list the exact source locations, and record measured N, fs, duration, units and dtype. Check metadata against the array. Create your experiment record and explain which settings are fixed versus provisional.

### Evidence and completion checks
A source inventory, exact sample counts and one correct time axis.

**Cell 6 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [46]:
# Imports
import sys
import json
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
import h5py
import gwosc
import matplotlib.pyplot as plt

from gwosc.datasets import find_datasets, run_segment, event_gps
from gwosc.timeline import get_segments
from gwosc.locate import get_urls


# Locating project root by finding both requirements.txt and notebooks folder
def find_project_root(start: Path) -> Path:
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ligo-parameter-inference repo.")

PROJECT_ROOT = find_project_root(Path.cwd())


# Experiment ID here is provisional and subject to change. Currently chosen to match other notebooks.
EXPERIMENT_ID = "manual_preprocessing_v1"

EXPERIMENT_ROOT = PROJECT_ROOT / "artifacts" / "manual_preprocessing_milestone" / EXPERIMENT_ID
DATA_DIR = EXPERIMENT_ROOT / "data"      # Folder owned by this notebook
RAW_CACHE_DIR = DATA_DIR / "raw_cache"   # downloaded GWOSC HDF5 files
NOISE_DIR = DATA_DIR / "noise"           # exported <noise_id>.npy arrays (Task 5)

for d in (DATA_DIR, RAW_CACHE_DIR, NOISE_DIR):
    d.mkdir(parents=True, exist_ok=True)


# Settings based on team contracts.
DETECTOR = "H1"
SAMPLE_RATE_HZ = 4096
RUN_DATASETS = ["O3a", "O4a"]
FINAL_WINDOW_SEC = 4.0
BAND_HZ = (35.0, 350.0)
TARGET_ORDER = ["chirp_mass", "mass_ratio", "chi_eff", "distance"]

# Provisional settings, subject to change if team agrees on it.
SEGMENT_DURATION_SEC = 512                
EVENT_BUFFER_SEC = 3600                    # exclude windows within 1 h of any catalogued event or HW injection
GWOSC_FILE_SEC = 4096                     # GWOSC files cover fixed 4096 s blocks
EDGE_MARGIN_SEC = 64                       # Accounting for possible artifact from padding from when data is downscaled
N_WINDOWS = {"O3a": 5, "O4a": 2}           # final pilot design (Task 4) ## REVISIT
O3A_SPLITS = ["train", "train", "train", "validation", "O3a_test"]   # chronological ## REVISIT


# Noting down software versions for the record
SOFTWARE_VERSIONS = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "h5py": h5py.__version__,
    "requests": requests.__version__,
    "gwosc": gwosc.__version__,
}


# Sanity checks
assert np.__version__.startswith("1."), f"NumPy {np.__version__} found; project requires <2.0" # Because torch 2.2 breaks with Numpy 2.x

print("Python executable:", sys.executable)
print("Project root:     ", PROJECT_ROOT)
print("Experiment root:  ", EXPERIMENT_ROOT)
print(json.dumps(SOFTWARE_VERSIONS, indent=2))

Python executable: /home/kennedy/ligo-parameter-inference/.venv/bin/python
Project root:      /home/kennedy/ligo-parameter-inference
Experiment root:   /home/kennedy/ligo-parameter-inference/artifacts/manual_preprocessing_milestone/manual_preprocessing_v1
{
  "python": "3.11.16",
  "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43",
  "numpy": "1.26.4",
  "pandas": "3.0.6",
  "h5py": "3.16.0",
  "requests": "2.34.2",
  "gwosc": "0.8.3"
}


In [47]:
# Looking up the data sources

# Find and confirm GWOSC dataset names for H1
available_runs = find_datasets(type="run", detector=DETECTOR)
print("Runs available for H1:", available_runs)

for dataset in RUN_DATASETS:
    assert dataset in available_runs, f"{dataset} not found; check your gwosc version"
print("Both dataset names confirmed.")
print('\n')



# Run boundaries from GWOSC to get GPS time of start and end for each run, expressed in days
RUN_BOUNDS = {}
for dataset in RUN_DATASETS:
    start, end = run_segment(dataset)
    RUN_BOUNDS[dataset] = (int(start), int(end))
    print(f"{dataset}: GPS {start} to {end}  ({(end - start) / 86400:.1f} days)")
print('\n')



# Retrieves interval for when H1 is public and data is available. 
H1_DATA_SEGMENTS = {}
for label, (start, end) in RUN_BOUNDS.items():
    segs = get_segments(f"{DETECTOR}_DATA", start, end)
    H1_DATA_SEGMENTS[label] = segs
    total_hours = sum(e - s for s, e in segs) / 3600
    longest = max(e - s for s, e in segs)
    print(f"{label}: {len(segs)} H1_DATA segments, {total_hours:.0f} h total, longest {longest} s")
print('\n')



# Probe which CBC / hardware-injection flag names this GWOSC host accepts.
# Names differ between runs, so try candidates and record which ones work.
candidate_flags = [f"{DETECTOR}_CBC_CAT1", f"{DETECTOR}_CBC_CAT2", f"{DETECTOR}_NO_CBC_HW_INJ"]
for label, (start, end) in RUN_BOUNDS.items():
    for flag in candidate_flags:
        try:
            segs = get_segments(flag, start, end)
            print(f"{label}: {flag} OK ({len(segs)} segments)")
        except Exception as exc:
            print(f"{label}: {flag} not available ({type(exc).__name__})")
print('\n')


# Returns number of real gravitational wave events in each run. We're trying to avoid them as we want pure noise for our injection.
EVENT_GPS = {}
for label, (start, end) in RUN_BOUNDS.items():
    names = find_datasets(type="event", segment=(start, end))
    times = sorted({float(event_gps(name)) for name in names})
    EVENT_GPS[label] = times
    print(f"{label}: {len(names)} event entries -> {len(times)} unique event times")

Runs available for H1: ['O1', 'O2', 'O3a', 'O3b', 'O4a', 'O4b', 'O4c1DiscC00', 'S5', 'S6']
Both dataset names confirmed.


O3a: GPS 1238166018 to 1253977218  (183.0 days)
O4a: GPS 1368195220 to 1389456018  (246.1 days)


O3a: 545 H1_DATA segments, 3116 h total, longest 138170 s
O4a: 823 H1_DATA segments, 3854 h total, longest 163354 s


O3a: H1_CBC_CAT1 OK (545 segments)
O3a: H1_CBC_CAT2 OK (629 segments)
O3a: H1_NO_CBC_HW_INJ OK (2 segments)
O4a: H1_CBC_CAT1 OK (823 segments)
O4a: H1_CBC_CAT2 OK (823 segments)
O4a: H1_NO_CBC_HW_INJ OK (2 segments)


O3a: 136 event entries -> 80 unique event times
O4a: 272 event entries -> 141 unique event times


In [48]:
# After digging, it turns out that the official start for O4a is later than what we currently have, so we'll be updating our run bounds.

O4A_OFFICIAL_START_GPS = 1368975618  # gwosc.org/O4/O4a: run start 2023-05-24 15:00 UTC
RUN_BOUNDS["O4a"] = (max(RUN_BOUNDS["O4a"][0], O4A_OFFICIAL_START_GPS), RUN_BOUNDS["O4a"][1])

# We'll also be recomputing the segments and events

segs = get_segments(f"{DETECTOR}_DATA", RUN_BOUNDS["O4a"][0], RUN_BOUNDS["O4a"][1])
H1_DATA_SEGMENTS["O4a"] = segs
total_hours = sum(e - s for s, e in segs) / 3600
longest = max(e - s for s, e in segs)
print(f"O4a: {len(segs)} H1_DATA segments, {total_hours:.0f} h total, longest {longest} s")

names = find_datasets(type="event", segment=(RUN_BOUNDS["O4a"][0], RUN_BOUNDS["O4a"][1]))
times = sorted({float(event_gps(name)) for name in names})
EVENT_GPS["O4a"] = times
print(f"O4a: {len(names)} event entries -> {len(times)} unique event times")

O4a: 814 H1_DATA segments, 3836 h total, longest 163354 s
O4a: 269 event entries -> 139 unique event times


In [49]:
# Only 2 runs each for NO_CBC_HW_INJ. Checking how much it actually covers for the runs.

for run, (start, end) in RUN_BOUNDS.items():
    segs = get_segments(f"{DETECTOR}_NO_CBC_HW_INJ", start, end)
    print(run, f"{sum(e - s for s, e in segs) / 3600:.0f} h", segs)

O3a 4392 h [(1238166018, 1251662270), (1251662279, 1253977218)]
O4a 5689 h [(1368975618, 1388879319), (1388879412, 1389456018)]


In [50]:
# Reconfirming event counts, dropping near duplicates (dropping the version suffix) to get a more accurate count

for run, (start, end) in RUN_BOUNDS.items():
    names = find_datasets(type="event", segment=(start, end))
    base = {n.split("-v")[0] for n in names}
    print(run, len(base), "distinct events")

O3a 63 distinct events
O4a 138 distinct events


In [51]:
"""
Next, we will separate our noise data into distinct windows. We want data from when the H1 detector is recording (H1_DATA), which passes the quality vetos (H1_CBC_CAT2), and 
has no hardware injection (H1_NO_CBC_HW_INJ)
"""
def intersection_of_dataset(A, B):
    intervals = []
    i, j = 0, 0
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])
        hi = min(A[i][1],   B[j][1])
        if lo < hi: 
            intervals.append((lo, hi))
        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1 
    return intervals

H1_CBC_DATA = {"O3a": get_segments('H1_CBC_CAT2', RUN_BOUNDS['O3a'][0], RUN_BOUNDS['O3a'][1]), "O4a": get_segments('H1_CBC_CAT2', RUN_BOUNDS['O4a'][0], RUN_BOUNDS['O4a'][1])}
H1_CBC_DATA_NO_HARDWARE_INJECTION = {"O3a": get_segments('H1_NO_CBC_HW_INJ', RUN_BOUNDS['O3a'][0], RUN_BOUNDS['O3a'][1]), "O4a": get_segments('H1_NO_CBC_HW_INJ', RUN_BOUNDS['O4a'][0], RUN_BOUNDS['O4a'][1])}

VALID_INTERVALS = {}
VALID_INTERVALS["O3a"] = intersection_of_dataset(intersection_of_dataset(H1_DATA_SEGMENTS["O3a"], H1_CBC_DATA["O3a"]), H1_CBC_DATA_NO_HARDWARE_INJECTION["O3a"])
VALID_INTERVALS["O4a"] = intersection_of_dataset(intersection_of_dataset(H1_DATA_SEGMENTS["O4a"], H1_CBC_DATA["O4a"]), H1_CBC_DATA_NO_HARDWARE_INJECTION["O4a"])

In [52]:
print(f"The total hours of each resulting run is {sum([interval[1] - interval[0] for interval in VALID_INTERVALS['O3a']])/3600} and {sum([interval[1] - interval[0] for interval in VALID_INTERVALS['O4a']])/3600} for O3a and O4a respectively.")

The total hours of each resulting run is 3104.7325 and 3835.6111111111113 for O3a and O4a respectively.


In [53]:
"""
After that, we'll isolate noise data from the runs as windows, including a ledger of accepted and rejected runs.
"""


# Two gaps happened because of hardware injections, and so we're excluding them from our windows.
INJECTION_GAPS = {
    "O3a": [(1251662270, 1251662279)],
    "O4a": [(1388879319, 1388879412)],
}

EXCLUSIONS = {} # Constructing an exclusion list of gravitational events + hardware injection to obtain pure noise only.
for run in RUN_DATASETS:
    events = [(t, t) for t in EVENT_GPS[run]]          # Convert a point gravitaional event into a zero-length interval
    EXCLUSIONS[run] = sorted(events + INJECTION_GAPS[run])

def inside_allowed(w0, w1, allowed):
    """Returns True if [w0, w1) lies fully inside one allowed (a, b)."""
    for interval in allowed:
        if (w0 >= interval[0]) and (w1 <= interval[1]):
            return True
    return False

def near_exclusion(w0, w1, exclusions, buffer):
    """Returns True if [w0, w1) overlaps any exclusion widened by `buffer` on both sides."""
    for interval in exclusions:
            if (w0 < interval[1] + buffer) and (w1 > interval[0] - buffer):
                return True
    return False

# Cuts runs into regions so we get a more balanced set of windows from the runs in return
def spread_regions(start, end, k):
    """Split [start, end) into k equal regions; return (region_start, region_end, anchor)."""
    edges = [start + round(i * (end - start) / k) for i in range(k + 1)]
    return [(edges[i], edges[i + 1], (edges[i] + edges[i + 1]) // 2) for i in range(k)]

REGIONS = {run: spread_regions(*RUN_BOUNDS[run], N_WINDOWS[run]) for run in RUN_DATASETS} # Retrieve region interval and anchors for each run
ANCHOR_GPS = {run: [anchor for _, _, anchor in REGIONS[run]] for run in RUN_DATASETS} # And retrieve the anchors only from the regions


def search_region(run, region_index, region_end, anchor):
    """First valid window at or after `anchor` and before `region_end`; logs every candidate."""
    ledger = []
    block = -(-anchor // GWOSC_FILE_SEC)             # first file starting at/after the anchor, with ceiling applied
    while True:
        file_start = block * GWOSC_FILE_SEC
        w0 = file_start + EDGE_MARGIN_SEC
        w1 = w0 + SEGMENT_DURATION_SEC
        if w1 > region_end:
            ledger.append({"run": run, "region": region_index, "gps_start": None, "gps_end": None,
                           "file_index": None, "reason": "no valid window in region"})
            return None, ledger
        assert w1 <= file_start + GWOSC_FILE_SEC - EDGE_MARGIN_SEC   # inside one file, clear of edges

        # Check if our window is inside the allowed interval or near an exclusion point and rejects it. Otherwise, accept it.
        if not inside_allowed(w0, w1, VALID_INTERVALS[run]):
            reason = "not in DATA ∩ CAT2 ∩ NO_INJ"
        elif near_exclusion(w0, w1, EXCLUSIONS[run], EVENT_BUFFER_SEC):
            reason = "within buffer of event/injection"
        else:
            reason = "accepted"

        row = {"run": run, "region": region_index, "gps_start": w0, "gps_end": w1,
               "file_index": block, "reason": reason}
        ledger.append(row)
        if reason == "accepted":
            return row, ledger
        block += 1


SELECTED, LEDGER = [], []
for run in RUN_DATASETS:
    for i, (r_start, r_end, anchor) in enumerate(REGIONS[run]):
        row, led = search_region(run, i + 1, r_end, anchor)
        LEDGER += led
        if row is not None:
            row["split"] = O3A_SPLITS[i] if run == "O3a" else "O4a_test"
            row["group_id"] = f"{run}_g{i + 1:02d}"
            SELECTED.append(row)

SELECTED = pd.DataFrame(SELECTED)
LEDGER = pd.DataFrame(LEDGER)
display(SELECTED)
display(LEDGER.groupby(["run", "reason"]).size())

,run,region,gps_start,gps_end,file_index,reason,split,group_id
0,O3a,1,1239760960,1239761472,302676,accepted,train,O3a_g01
1,O3a,2,1242927168,1242927680,303449,accepted,train,O3a_g02
2,O3a,3,1246081088,1246081600,304219,accepted,train,O3a_g03
3,O3a,4,1249235008,1249235520,304989,accepted,validation,O3a_g04
4,O3a,5,1252397120,1252397632,305761,accepted,O3a_test,O3a_g05
5,O4a,1,1374097472,1374097984,335473,accepted,O4a_test,O4a_g01
6,O4a,2,1384337472,1384337984,337973,accepted,O4a_test,O4a_g02


run  reason                     
O3a  accepted                       5
     not in DATA ∩ CAT2 ∩ NO_INJ    9
O4a  accepted                       2
dtype: int64

In [54]:
# check that our results match our logic. No anchor exists within EVENT_BUFFER_SEC (1 hour here) from the nearest event
def nearest_exclusion_sec(w0, w1, exclusions):
    """Smallest gap in seconds between [w0, w1) and any exclusion interval (0 if overlapping)."""
    return min(max(x0 - w1, w0 - x1, 0) for x0, x1 in exclusions)

SELECTED["nearest_exclusion_h"] = [
    nearest_exclusion_sec(r.gps_start, r.gps_end, EXCLUSIONS[r.run]) / 3600
    for r in SELECTED.itertuples()
]
assert (SELECTED["nearest_exclusion_h"] >= EVENT_BUFFER_SEC / 3600).all()
display(SELECTED[["run", "group_id", "split", "gps_start", "nearest_exclusion_h"]])

,run,group_id,split,gps_start,nearest_exclusion_h
0,O3a,O3a_g01,train,1239760960,43.467278
1,O3a,O3a_g02,train,1242927168,15.664917
2,O3a,O3a_g03,train,1246081088,9.078722
3,O3a,O3a_g04,validation,1249235008,44.525750
4,O3a,O3a_g05,O3a_test,1252397120,63.724611
5,O4a,O4a_g01,O4a_test,1374097472,10.926222
6,O4a,O4a_g02,O4a_test,1384337472,1.081111


In [55]:
"""
Locating, downloading, reading and verifying the raw GWOSC strain for the handoff windows.

For each selected window we
  - Ask GWOSC which 4 kHz HDF5 file covers it
  - Download that file once into RAW_CACHE_DIR and record its size and MD5 checksum
  - Read the file's own metadata, check it against the stored array, and crop our 512 s window
  - Convert the window start from GPS to UTC and cross-check against the file's UTC metadata
"""

HANDOFF_GROUPS = ["O3a_g01", "O4a_g01"]   # first handoff; set to list(SELECTED.group_id) to fetch all 7
GPS_EPOCH = datetime(1980, 1, 6, tzinfo=timezone.utc)
GPS_MINUS_UTC_SEC = 18                     # leap seconds since 1980; valid for GPS times after 2017-01-01
LAST_LEAP_SECOND_GPS = 1167264018          # GPS time of 2017-01-01 00:00:00 UTC


def gps_to_utc(gps):
    """GPS seconds -> UTC datetime. GPS does not apply leap seconds, so subtract the 18 s offset."""
    assert gps >= LAST_LEAP_SECOND_GPS, "18 s offset only valid after 2017-01-01"
    return GPS_EPOCH + timedelta(seconds=float(gps) - GPS_MINUS_UTC_SEC)


def to_str(value):
    """HDF5 strings come back as bytes, numpy bytes or 1-element arrays; normalise to str."""
    if isinstance(value, np.ndarray):
        value = value.item() if value.size == 1 else value.tolist()
    if isinstance(value, (bytes, np.bytes_)):
        value = value.decode()
    return value


# Locating the file
def locate_file(gps_start, gps_end, file_index):
    urls = get_urls(DETECTOR, gps_start, gps_end, sample_rate=SAMPLE_RATE_HZ, format="hdf5")
    assert len(urls) == 1, f"expected exactly one file for [{gps_start}, {gps_end}), got {len(urls)}: {urls}"
    url = urls[0]
    file_start = file_index * GWOSC_FILE_SEC
    filename = url.rsplit("/", 1)[-1]
    # GWOSC filenames look like H-H1_GWOSC_O3a_4KHZ_R1-<file_start>-4096.hdf5
    assert f"-{file_start}-{GWOSC_FILE_SEC}." in filename, f"{filename} does not start at {file_start}"
    return url, filename


# Downloading file, with checksum
def download_file(url, filename, chunk_bytes=1 << 20):
    """Download to RAW_CACHE_DIR unless already present. Returns (path, md5, n_bytes, downloaded_now)."""
    path = RAW_CACHE_DIR / filename
    md5 = hashlib.md5()
    if path.exists():
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_bytes), b""):
                md5.update(chunk)
        return path, md5.hexdigest(), path.stat().st_size, False

    partial = path.with_suffix(path.suffix + ".part")   # never leave a half-written file under the real name
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(partial, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_bytes):
                f.write(chunk)
                md5.update(chunk)
    partial.rename(path)
    return path, md5.hexdigest(), path.stat().st_size, True


# Reading and verifying metadata against the array, crop
def read_and_crop(path, gps_start, gps_end, file_index):
    with h5py.File(path, "r") as f:
        strain = f["strain/Strain"]
        attrs = {k: to_str(v) for k, v in strain.attrs.items()}
        meta = {k: to_str(f["meta"][k][()]) for k in f["meta"].keys()}

        x_start = float(attrs["Xstart"])
        x_spacing = float(attrs["Xspacing"])
        n_points_attr = int(attrs["Npoints"])
        n_points_array = strain.shape[0]
        dtype_in_file = str(strain.dtype)          # read now; the dataset is unusable once the file closes
        fs_measured = 1.0 / x_spacing

        # Checking metadata vs. array
        checks = {
            "Npoints attr == array length": n_points_attr == n_points_array,
            "1 / Xspacing == 4096 Hz": np.isclose(fs_measured, SAMPLE_RATE_HZ),
            "Npoints * Xspacing == meta Duration": np.isclose(n_points_array * x_spacing, float(meta["Duration"])),
            "Xstart == meta GPSstart": np.isclose(x_start, float(meta["GPSstart"])),
            "Xstart == file_index * 4096": np.isclose(x_start, file_index * GWOSC_FILE_SEC),
            "meta Detector == H1": str(meta.get("Detector")) == DETECTOR,
            "array is 1-D": strain.ndim == 1,
        }

        # Crop: sample k of the file is at Xstart + k / fs, so our window starts at index i0.
        i0_exact = (gps_start - x_start) * SAMPLE_RATE_HZ
        i0 = int(round(i0_exact))
        n_window = (gps_end - gps_start) * SAMPLE_RATE_HZ
        i1 = i0 + n_window
        checks["window start falls exactly on a sample"] = abs(i0_exact - i0) < 1e-6
        checks["window lies inside the file"] = 0 <= i0 and i1 <= n_points_array
        window = strain[i0:i1].astype(np.float64)   # read only our 512 s, not the whole file

        # Data-quality and injection bitmasks are 1 Hz vectors; keep the seconds covering our window
        # (decoded and judged in the Task 2 audit).
        s0, s1 = int(gps_start - x_start), int(gps_end - x_start)
        dq = {"dq_mask": f["quality/simple/DQmask"][s0:s1],
              "dq_names": [to_str(n) for n in f["quality/simple/DQShortnames"][()]],
              "inj_mask": f["quality/injections/Injmask"][s0:s1],
              "inj_names": [to_str(n) for n in f["quality/injections/InjShortnames"][()]]}

    # Time axis of the crop, and its consistency with the half-open [gps_start, gps_end) contract.
    t0 = x_start + i0 / SAMPLE_RATE_HZ
    t_last = t0 + (window.size - 1) / SAMPLE_RATE_HZ
    checks["crop length == 512 s * 4096 Hz"] = window.size == SEGMENT_DURATION_SEC * SAMPLE_RATE_HZ
    checks["crop t0 == gps_start"] = np.isclose(t0, gps_start)
    checks["t0 + N / fs == gps_end (exclusive)"] = np.isclose(t0 + window.size / SAMPLE_RATE_HZ, gps_end)

    info = {"file_gps_start": x_start, "file_duration_sec": float(meta["Duration"]),
            "file_npoints": n_points_array, "fs_hz": fs_measured, "dtype_in_file": dtype_in_file,
            "x_units": attrs.get("Xunits", "not stated"), "file_utc_start": meta.get("UTCstart", "not stated"),
            "crop_i0": i0, "crop_n": window.size, "crop_t0": t0, "crop_t_last": t_last,
            "finite_fraction": float(np.isfinite(window).mean())}
    return window, info, checks, dq, attrs, meta


# Running functions for thehandoff
RAW_WINDOWS, RAW_DQ, INVENTORY, CHECK_ROWS = {}, {}, [], []

for row in SELECTED[SELECTED.group_id.isin(HANDOFF_GROUPS)].itertuples():
    gps_start, gps_end, file_index = int(row.gps_start), int(row.gps_end), int(row.file_index)

    url, filename = locate_file(gps_start, gps_end, file_index)
    path, md5, n_bytes, downloaded_now = download_file(url, filename)
    window, info, checks, dq, attrs, meta = read_and_crop(path, gps_start, gps_end, file_index)

    RAW_WINDOWS[row.group_id] = window
    RAW_DQ[row.group_id] = dq
    CHECK_ROWS += [{"group_id": row.group_id, "check": name, "passed": bool(ok)} for name, ok in checks.items()]

    utc_window = gps_to_utc(gps_start)
    utc_file_computed = gps_to_utc(info["file_gps_start"])
    INVENTORY.append({
        "group_id": row.group_id, "run": row.run, "split": row.split, "detector": DETECTOR,
        "gps_start": gps_start, "gps_end": gps_end, "utc_start": utc_window.isoformat(),
        "source_url": url, "file_md5": md5, "file_mb": round(n_bytes / 1e6, 1),
        "downloaded_now": downloaded_now,
        **info,
        "file_utc_computed": utc_file_computed.isoformat(),
        # Independent check of the GPS->UTC conversion against GWOSC's own UTC label for the file.
        # Informational (not asserted) in case GWOSC formats UTCstart differently.
        "utc_matches_file_meta": str(info["file_utc_start"])[:19] == utc_file_computed.strftime("%Y-%m-%dT%H:%M:%S"),
        "window_dtype": str(window.dtype),
        "units": "strain (dimensionless)",
    })
    print(f"{row.group_id}: {filename} ({n_bytes / 1e6:.1f} MB, md5 {md5}) "
          f"{'downloaded' if downloaded_now else 'from cache'}")
    print(f"   file strain attrs: {attrs}")

INVENTORY = pd.DataFrame(INVENTORY)
CHECKS = pd.DataFrame(CHECK_ROWS)

display(INVENTORY.T)                                            # one column per window, easier to read
display(CHECKS.pivot(index="check", columns="group_id", values="passed"))
assert CHECKS.passed.all(), "a metadata/array check failed; see the table above"
print("All metadata-vs-array and time-axis checks passed.")

O3a_g01: H-H1_GWOSC_O3a_4KHZ_R1-1239760896-4096.hdf5 (129.5 MB, md5 28852e47c7dc261d5679c775f90e59c4) from cache
   file strain attrs: {'Npoints': 16777216, 'Xlabel': 'GPS time', 'Xspacing': 0.000244140625, 'Xstart': 1239760896, 'Xunits': 'second', 'Ylabel': 'Strain', 'Yunits': ''}
O4a_g01: H-H1_GWOSC_O4a_4KHZ_R1-1374097408-4096.hdf5 (104.3 MB, md5 66dece51a0fb6e0808bd6a20e823a6cd) from cache
   file strain attrs: {'Npoints': 16777216, 'Xlabel': 'GPS time', 'Xspacing': 0.000244140625, 'Xstart': 1374097408, 'Xunits': 'second', 'Ylabel': 'Strain', 'Yunits': ''}


,0,1
group_id,O3a_g01,O4a_g01
run,O3a,O4a
split,train,O4a_test
detector,H1,H1
gps_start,1239760960,1374097472
gps_end,1239761472,1374097984
utc_start,2019-04-20T02:02:22+00:00,2023-07-22T21:44:14+00:00
source_url,https://gwosc.org/archive/data/O3a_4KHZ_R1/123...,https://gwosc.org/archive/data/O4a_4KHZ_R1/137...
file_md5,28852e47c7dc261d5679c775f90e59c4,66dece51a0fb6e0808bd6a20e823a6cd
file_mb,129.5,104.3


group_id,O3a_g01,O4a_g01
check,,
1 / Xspacing == 4096 Hz,True,True
Npoints * Xspacing == meta Duration,True,True
Npoints attr == array length,True,True
Xstart == file_index * 4096,True,True
Xstart == meta GPSstart,True,True
array is 1-D,True,True
crop length == 512 s * 4096 Hz,True,True
crop t0 == gps_start,True,True
meta Detector == H1,True,True


All metadata-vs-array and time-axis checks passed.


**Cell 8**

## Task 2: Acquire and audit raw backgrounds

### Background knowledge
Real noise can contain gaps, invalid samples, transients and known astrophysical events. Catalog exclusions and quality flags reduce contamination but do not prove a segment is glitch-free. Missing strain is not a categorical value to encode.

### Your task
Read/fetch candidate segments yourself. Check shape, finite fraction, mean/std/min/max and available data-quality information. Explain your rejection rules and event exclusion policy. Retain an accepted/rejected ledger. Reject invalid data rather than silently filling gaps; if you choose interpolation, justify why it is scientifically suitable.

### Evidence and completion checks
QC table with actual before/after counts, reasons and a clear limitation when quality flags are unavailable.

**Cell 9 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [60]:
print(RAW_DQ["O3a_g01"]["dq_names"])
print(RAW_DQ["O3a_g01"]["inj_names"])

['DATA', 'CBC_CAT1', 'CBC_CAT2', 'CBC_CAT3', 'BURST_CAT1', 'BURST_CAT2', 'BURST_CAT3']
['NO_CBC_HW_INJ', 'NO_BURST_HW_INJ', 'NO_DETCHAR_HW_INJ', 'NO_CW_HW_INJ', 'NO_STOCH_HW_INJ']


In [64]:
print(RAW_WINDOWS.items())

dict_items([('O3a_g01', array([-4.82404640e-19, -4.77955394e-19, -4.55584711e-19, ...,
       -2.50127582e-19, -2.25903809e-19, -2.08094657e-19])), ('O4a_g01', array([-2.28143525e-18, -2.13173157e-18, -2.28979025e-18, ...,
        7.61789092e-19,  8.33746708e-19,  7.40242064e-19]))])


In [ ]:
"""
We will next audit the fetched raw windows and decide accept/reject with these rules.
Rules:
  R1  shape:     1-D, exactly 512 s x 4096 Hz samples
  R2  finite:    every sample finite. Invalid data is rejected, never interpolated
  R3  DQ:        DATA, CBC_CAT1, CBC_CAT2 set for every second of the window
  R4  injection: NO_CBC_HW_INJ and NO_BURST_HW_INJ set for every second
                 (NO_CW_HW_INJ not required: CW injections are continuous, very weak, and narrow-band. Will likely be 0 everywhere anyway)

"""

REQUIRED_DQ_BITS = ["DATA", "CBC_CAT1", "CBC_CAT2"]
REQUIRED_INJ_BITS = ["NO_CBC_HW_INJ", "NO_BURST_HW_INJ"]
EXPECTED_N = SEGMENT_DURATION_SEC * SAMPLE_RATE_HZ

# Function to check if the proper flags exist, using bitmasking.
def bit_fraction(mask, names, bit_name):
    """Fraction of seconds in `mask` where the bit called `bit_name` is set (1.0 = whole window).
    Returns None if this file has no bit with that name."""

    if bit_name not in names:
        return None
    bit_name_pos = names.index(bit_name)
    is_set = ((mask >> bit_name_pos) & 1) == 1
    return float(is_set.mean())
    

def window_stats(x):
    """Descriptive statistics of one raw window."""

    stats = {}
    stats["n_samples"] = int(x.size)
    stats["ndim"] = int(x.ndim)
    stats["finite_fraction"] = float(np.isfinite(x).mean())
    stats["mean"] = float(np.mean(x))
    stats["std"] = float(np.std(x))
    stats["min"] = float(np.min(x))
    stats["max"] = float(np.max(x))
    stats["max_abs"] = float(np.max(np.abs(x)))
    return stats


# Function where we perform our rejection decision
def qc_decision(stats, dq):
    """Apply R1-R4 in order. Returns (status, reason, bit_fractions)."""
    fractions = {name: bit_fraction(dq["dq_mask"], dq["dq_names"], name) for name in REQUIRED_DQ_BITS}
    fractions |= {name: bit_fraction(dq["inj_mask"], dq["inj_names"], name) for name in REQUIRED_INJ_BITS}
    
    # R1
    if stats["ndim"] != 1 or stats["n_samples"] != EXPECTED_N:
        return "rejected", f"R1: shape ndim={stats['ndim']}, n={stats['n_samples']}, expected 1-D with {EXPECTED_N}", fractions

    # R2
    if stats["finite_fraction"] < 1.0:
        return "rejected", f"R2: only {stats['finite_fraction']:.4%} of samples finite", fractions

    # R3
    for name in REQUIRED_DQ_BITS:
        frac = fractions[name]
        if frac is None:
            return "rejected", f"R3: DQ flag {name} not present in file", fractions
        if frac < 1.0:
            return "rejected", f"R3: {name} set for only {frac:.1%} of window", fractions

    # R4
    for name in REQUIRED_INJ_BITS:
        frac = fractions[name]
        if frac is None:
            return "rejected", f"R4: injection flag {name} not present in file", fractions

        if frac < 1.0:
            return "rejected", f"R4: {name} set for only {frac:.1%} of window (hardware injection present)", fractions

    return "accepted", "passed R1-R4", fractions


In [97]:
"""
Build and return the QC and accounting table

Acquisition accounting: where every candidate went, from selection through QC.
Each candidate ends in exactly one place (rejected with a reason, accepted, or pending),
and the asserts below fail loudly if the counts stop adding up.
"""

QC_ROWS = []
for group_id, x in RAW_WINDOWS.items():
    stats = window_stats(x)
    status, reason, fractions = qc_decision(stats, RAW_DQ[group_id])
    QC_ROWS.append({"group_id": group_id, **stats, **{f"frac_{k}": v for k, v in fractions.items()},
                    "quality_status": status, "quality_reason": reason})

QC = pd.DataFrame(QC_ROWS).merge(SELECTED[["group_id", "run", "split", "gps_start", "gps_end"]], on="group_id")
display(QC.T)



SELECTION_REJECT_REASONS = ["not in DATA ∩ CAT2 ∩ NO_INJ", "within buffer of event/injection"]
NO_WINDOW_REASON = "no valid window in region"

def per_run(values):
    """Turn a {run: count} mapping or Series into integers for every run, with 0 where a run is absent."""
    return pd.Series(values, dtype="float64").reindex(RUN_DATASETS, fill_value=0).fillna(0).astype(int)


# Selection stage (from LEDGER)
candidates = LEDGER[LEDGER["reason"] != NO_WINDOW_REASON]           # rows that are real candidate windows
checked = per_run(candidates.groupby("run").size())
reason_counts = {r: per_run(candidates[candidates["reason"] == r].groupby("run").size())
                 for r in SELECTION_REJECT_REASONS}
selection_accepted = per_run(candidates[candidates["reason"] == "accepted"].groupby("run").size())
regions_without_window = per_run(LEDGER[LEDGER["reason"] == NO_WINDOW_REASON].groupby("run").size())
passed_selection = per_run(SELECTED.groupby("run").size())

# Fetch stage (from RAW_WINDOWS)
fetched = per_run(SELECTED[SELECTED["group_id"].isin(list(RAW_WINDOWS))].groupby("run").size())
pending = passed_selection - fetched

# QC stage (from QC)
qc_rejected = per_run(QC[QC["quality_status"] == "rejected"].groupby("run").size())
qc_accepted_rows = QC[QC["quality_status"] == "accepted"]
qc_accepted = per_run(qc_accepted_rows.groupby("run").size())
accepted_samples = per_run(qc_accepted_rows.groupby("run")["n_samples"].sum())
accepted_seconds = accepted_samples // SAMPLE_RATE_HZ

ACCOUNTING = pd.DataFrame({
    "candidates checked (selection)": checked,
    **{f"  - rejected: {r}": reason_counts[r] for r in SELECTION_REJECT_REASONS},
    "  = passed selection (source groups)": passed_selection,
    "regions with no valid window": regions_without_window,
    "fetched for this handoff": fetched,
    "not yet fetched (pending)": pending,
    "  - rejected by QC (R1-R4)": qc_rejected,
    "  = accepted by QC": qc_accepted,
    "accepted raw seconds": accepted_seconds,
    "accepted raw samples": accepted_samples,
}).T
ACCOUNTING.index.name = "stage"

# Every count must reconcile; nothing may disappear between stages.
assert (checked == sum(reason_counts.values()) + selection_accepted).all(), "selection counts do not add up"
assert (selection_accepted == passed_selection).all(), "LEDGER accepted rows != SELECTED rows"
assert (pending >= 0).all(), "fetched a window that did not pass selection"
assert (fetched == qc_rejected + qc_accepted).all(), "QC counts do not add up"
assert set(LEDGER["reason"]) <= set(SELECTION_REJECT_REASONS) | {"accepted", NO_WINDOW_REASON}, \
    f"unexpected ledger reason(s): {set(LEDGER['reason'])}"

display(ACCOUNTING)
print("Selection rejection reasons:")
display(candidates.groupby(["run", "reason"]).size())
print("QC outcomes:")
display(QC.groupby(["run", "quality_status", "quality_reason"]).size())

# Save both tables as evidence (Role 4 audits these; repo-relative under this experiment's data/ folder)
ACCOUNTING.to_csv(DATA_DIR / "acquisition_accounting.csv")
QC.to_csv(DATA_DIR / "qc_windows.csv", index=False)
print("Saved:", (DATA_DIR / "acquisition_accounting.csv").relative_to(PROJECT_ROOT),
      "and", (DATA_DIR / "qc_windows.csv").relative_to(PROJECT_ROOT))

,0,1
group_id,O3a_g01,O4a_g01
n_samples,2097152,2097152
ndim,1,1
finite_fraction,1.0,1.0
mean,-0.0,-0.0
std,0.0,0.0
min,-0.0,-0.0
max,0.0,0.0
max_abs,0.0,0.0
frac_DATA,1.0,1.0


run,O3a,O4a
stage,,
candidates checked (selection),14,2
- rejected: not in DATA ∩ CAT2 ∩ NO_INJ,9,0
- rejected: within buffer of event/injection,0,0
= passed selection (source groups),5,2
regions with no valid window,0,0
fetched for this handoff,1,1
not yet fetched (pending),4,1
- rejected by QC (R1-R4),0,0
= accepted by QC,1,1


Selection rejection reasons:


run  reason                     
O3a  accepted                       5
     not in DATA ∩ CAT2 ∩ NO_INJ    9
O4a  accepted                       2
dtype: int64

QC outcomes:


run  quality_status  quality_reason
O3a  accepted        passed R1-R4      1
O4a  accepted        passed R1-R4      1
dtype: int64

Saved: artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/acquisition_accounting.csv and artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/qc_windows.csv


**Cell 11**

## Task 3: Inspect raw distributions and time behavior

### Background knowledge
A histogram reveals asymmetry or heavy tails; a full time plot may hide short transients. Blockwise mean/std reveal changing scale. Raw colored noise is not expected to have equal power at every frequency.

### Your task
Plot full time series, a short zoom, a histogram and blockwise statistics with time in seconds and strain on the correct axes. Compare runs when both are available. Interpret at least two patterns; do not claim stationarity from one plot alone.

### Evidence and completion checks
At least three properly labeled plots and short interpretations.

**Cell 12 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 14**

## Task 4: Freeze source groups and the split

### Background knowledge
Injections sharing the same background can share noise patterns. Randomly splitting their rows can exaggerate generalization. Group by original nonoverlapping GPS intervals before simulation; reserve validation for selection and tests for evaluation.

### Your task
Assign each accepted source group one split. Check GPS overlap, prohibit group reuse across O3a splits, and keep O4a test-only. Export the split manifest. Record intended example counts; identify that actual counts will be measured after injection and preprocessing.

### Evidence and completion checks
A group-disjoint split audit and source-duration/sample-count table per run and split.

**Cell 15 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 17**

## Task 5: Export one early handoff, then the pilot

### Background knowledge
A downstream handoff is a stable data contract. Files need explicit provenance so another person can inspect them without knowing your variable names.

### Your task
Export raw .npy files and the manifest. Ask Roles 2 and 3 to verify one row and array. Correct your own notebook from their feedback; then extend the pilot. Record source hashes if practical and versions/seeds.

### Evidence and completion checks
Role 2 and Role 3 can load one segment and verify fs, units, noise_id and split without importing your code.

**Cell 18 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

In [98]:
"""
First handoff. Export the QC-accepted raw windows and their provenance for Roles 2-4.

Writes, under artifacts/manual_preprocessing_milestone/<EXPERIMENT_ID>/data/:
  noise/<noise_id>.npy       1-D float64 strain, finite, no pickling (one per accepted window)
  noise_manifest.csv         one row per fetched window, exactly the contract columns
  source_inventory.csv       per-window provenance: URL, file MD5, UTC, file metadata (Steps G-J)
  selection_ledger.csv       every candidate considered in Step F, with its reason
  experiment.json            shared settings, selection rule, planned groups/splits, counts, versions
Then reloads everything the way a consumer would (no imports from this notebook) as a handoff check.
"""

OWNER = "Kennedy Goliman (Role 1)"
MANIFEST_COLUMNS = ["noise_id", "group_id", "run", "detector", "gps_start", "gps_end", "sample_rate_hz",
                    "n_samples", "strain_path", "split", "source_url_or_key", "quality_status", "quality_reason"]


def make_noise_id(run, gps_start):
    """Stable, unique ID: detector, run and first GPS second of the window."""
    return f"{DETECTOR}_{run}_{int(gps_start)}"


def repo_relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()


def save_array_or_validate(path, array):
    """Write the array; if a file already exists it must be identical, otherwise refuse to overwrite."""
    if path.exists():
        existing = np.load(path, allow_pickle=False)
        if existing.shape != array.shape or not np.array_equal(existing, array):
            raise FileExistsError(f"{path} exists with different contents; refusing to overwrite a handoff file")
        return "unchanged"
    np.save(path, array, allow_pickle=False)
    return "written"


def to_jsonable(obj):
    """json.dump helper: numpy scalars/arrays and Paths -> plain Python."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return obj.as_posix()
    raise TypeError(f"not JSON serialisable: {type(obj)}")


# Arrays + manifest
qc_by_group = QC.set_index("group_id")
inv_by_group = INVENTORY.set_index("group_id")
sel_by_group = SELECTED.set_index("group_id")

manifest_rows = []
for group_id, window in RAW_WINDOWS.items():
    sel, qc, inv = sel_by_group.loc[group_id], qc_by_group.loc[group_id], inv_by_group.loc[group_id]
    noise_id = make_noise_id(sel["run"], sel["gps_start"])
    strain_path = NOISE_DIR / f"{noise_id}.npy"

    if qc["quality_status"] == "accepted":
        action = save_array_or_validate(strain_path, np.ascontiguousarray(window, dtype=np.float64))
        path_field = repo_relative(strain_path)
    else:
        action, path_field = "not exported (QC rejected)", ""     # rejected windows keep their row, but no array

    manifest_rows.append({
        "noise_id": noise_id, "group_id": group_id, "run": sel["run"], "detector": DETECTOR,
        "gps_start": int(sel["gps_start"]), "gps_end": int(sel["gps_end"]),   # gps_end is exclusive
        "sample_rate_hz": SAMPLE_RATE_HZ, "n_samples": int(window.size),
        "strain_path": path_field, "split": sel["split"], "source_url_or_key": inv["source_url"],
        "quality_status": qc["quality_status"], "quality_reason": qc["quality_reason"],
    })
    print(f"{noise_id}: {action}")

NOISE_MANIFEST = pd.DataFrame(manifest_rows, columns=MANIFEST_COLUMNS)
NOISE_MANIFEST.to_csv(DATA_DIR / "noise_manifest.csv", index=False)

# Provenance tables that back up the manifest
INVENTORY.assign(noise_id=[make_noise_id(r, g) for r, g in zip(INVENTORY["run"], INVENTORY["gps_start"])]) \
         .to_csv(DATA_DIR / "source_inventory.csv", index=False)
LEDGER.to_csv(DATA_DIR / "selection_ledger.csv", index=False)


# experiment.json file
experiment = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_id_status": "provisional until all four role owners confirm",
    "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "owner": OWNER,
    "fixed_settings": {
        "detector": DETECTOR, "sample_rate_hz": SAMPLE_RATE_HZ, "runs": RUN_DATASETS,
        "final_window_sec": FINAL_WINDOW_SEC, "band_hz": list(BAND_HZ), "target_order": TARGET_ORDER,
        "strain_units": "dimensionless strain", "gps_end_convention": "exclusive",
    },
    "provisional_settings": {
        "segment_duration_sec": SEGMENT_DURATION_SEC, "edge_margin_sec": EDGE_MARGIN_SEC,
        "event_buffer_sec": EVENT_BUFFER_SEC, "n_windows_per_run": N_WINDOWS, "o3a_split_order": O3A_SPLITS,
    },
    "source": {
        "archive": "GWOSC (https://gwosc.org)", "file_format": "HDF5, 4096 Hz, 4096 s files",
        "run_bounds_gps": {run: list(RUN_BOUNDS[run]) for run in RUN_DATASETS},
        "run_bounds_note": f"O4a start trimmed to the official run start {O4A_OFFICIAL_START_GPS} "
                           "(GWOSC release includes earlier data)",
        "segment_flags_required": ["H1_DATA", "H1_CBC_CAT2", "H1_NO_CBC_HW_INJ"],
        "hardware_injection_intervals_gps": INJECTION_GAPS,
        "n_event_times_excluded": {run: len(EVENT_GPS[run]) for run in RUN_DATASETS},
    },
    "selection_rule": (
        "Each run is split into N equal time regions with an anchor at each region's midpoint. From the anchor, "
        "candidate windows start EDGE_MARGIN_SEC after each successive GWOSC file boundary; the first candidate "
        "fully inside DATA∩CAT2∩NO_CBC_HW_INJ and at least EVENT_BUFFER_SEC from every catalogued event and "
        "hardware injection is accepted. One window per region; O3a splits assigned chronologically; all O4a "
        "windows are O4a_test."
    ),
    "anchors_gps": ANCHOR_GPS,
    "planned_groups": SELECTED[["group_id", "run", "split", "gps_start", "gps_end"]].to_dict(orient="records"),
    "qc_rules": ["R1 1-D, 512 s x 4096 Hz", "R2 all samples finite (no interpolation)",
                 "R3 DATA, CBC_CAT1, CBC_CAT2 set every second", "R4 NO_CBC_HW_INJ, NO_BURST_HW_INJ set every second"],
    "counts": ACCOUNTING.to_dict(),
    "outputs": {name: repo_relative(DATA_DIR / name) for name in
                ["noise_manifest.csv", "source_inventory.csv", "selection_ledger.csv",
                 "acquisition_accounting.csv", "qc_windows.csv"]} | {"noise_dir": repo_relative(NOISE_DIR)},
    "software": SOFTWARE_VERSIONS,
}
with open(DATA_DIR / "experiment.json", "w") as f:
    json.dump(experiment, f, indent=2, default=to_jsonable)


# Handoff check. Assert that the csv file matches the format we want
check = pd.read_csv(DATA_DIR / "noise_manifest.csv")
assert list(check.columns) == MANIFEST_COLUMNS, "manifest columns differ from the contract"
assert check["noise_id"].is_unique, "noise_id must be unique"
assert (check.loc[check["run"] == "O4a", "split"] == "O4a_test").all(), "O4a must be test-only"
assert (check.groupby("group_id")["split"].nunique() == 1).all(), "a group appears in more than one split"
assert (check["gps_end"] - check["gps_start"] == check["n_samples"] / check["sample_rate_hz"]).all()

accepted = check[check["quality_status"].str.lower() == "accepted"]
assert len(accepted) > 0, "no accepted rows to hand off"
for row in accepted.itertuples():
    arr = np.load(PROJECT_ROOT / row.strain_path, allow_pickle=False)
    assert arr.ndim == 1 and arr.size == row.n_samples, f"{row.noise_id}: shape {arr.shape} vs n_samples {row.n_samples}"
    assert arr.dtype == np.float64 and np.isfinite(arr).all(), f"{row.noise_id}: dtype/finite check failed"
    assert row.sample_rate_hz == SAMPLE_RATE_HZ
with open(DATA_DIR / "experiment.json") as f:
    assert json.load(f)["experiment_id"] == EXPERIMENT_ID

display(check)
print(f"Handoff check passed: {len(accepted)} accepted array(s) load as 1-D finite float64 with matching n_samples.")
print("\nCopy these to the team's shared folder at the same relative paths:")
for p in sorted([*DATA_DIR.glob("*.csv"), DATA_DIR / "experiment.json", *NOISE_DIR.glob("*.npy")]):
    print("  ", repo_relative(p))

H1_O3a_1239760960: written
H1_O4a_1374097472: written


,noise_id,group_id,run,detector,gps_start,gps_end,sample_rate_hz,n_samples,strain_path,split,source_url_or_key,quality_status,quality_reason
0,H1_O3a_1239760960,O3a_g01,O3a,H1,1239760960,1239761472,4096,2097152,artifacts/manual_preprocessing_milestone/manua...,train,https://gwosc.org/archive/data/O3a_4KHZ_R1/123...,accepted,passed R1-R4
1,H1_O4a_1374097472,O4a_g01,O4a,H1,1374097472,1374097984,4096,2097152,artifacts/manual_preprocessing_milestone/manua...,O4a_test,https://gwosc.org/archive/data/O4a_4KHZ_R1/137...,accepted,passed R1-R4


Handoff check passed: 2 accepted array(s) load as 1-D finite float64 with matching n_samples.

Copy these to the team's shared folder at the same relative paths:
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/acquisition_accounting.csv
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/experiment.json
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/noise/H1_O3a_1239760960.npy
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/noise/H1_O4a_1374097472.npy
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/noise_manifest.csv
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/qc_windows.csv
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/selection_ledger.csv
   artifacts/manual_preprocessing_milestone/manual_preprocessing_v1/data/source_inventory.csv


**Cell 20**

## Task 6: Prepare your milestone evidence

### Background knowledge
Data claims need separate counts for raw time samples, quality-screened segments and labeled examples. You can report raw counts now and receive example counts from the other roles.

### Your task
Write 150 words on source, size, QC, temporal patterns and data challenges. Include a plot caption and source link. Update your contribution log and notebook numbering. State what has actually run and what remains planned.

### Evidence and completion checks
A raw-data evidence bundle and a contribution entry for this notebook alone.

**Cell 21 is your empty work cell.** Write your own implementation if this task needs code. For planning/writing-only tasks, leave it empty and complete your response below. Do not train a model merely to fill a cell.

**Cell 23**

## Milestone completion and handoff

Before marking your role complete, show the teammate consuming your artifacts one real record and its provenance. Give them a settings file, retained/rejected counts and a short interpretation, not just an array. Check sample_id, noise_id, group_id, run, split and shape at the handoff. Save outputs under your role-owned folder for the same experiment_id.

Deliver your completed notebook, a contribution log, 100-150 words of findings, figures with descriptive titles/axes/units/legends/captions and an explicit list of unfinished checks. Do not copy another person's implementation into your notebook. Preserve the older notebooks and existing PDFs.

**Stop at this checkpoint:** dataset preparation, validated preprocessing and EDA, exact allocation tables and the three-algorithm methodology plan. Full linear/Ridge/SNPE training, final model ranking, posterior calibration results and 800-per-run manual regeneration are not mandatory tasks in this set. Feature importance remains an acknowledged rubric gap if deferred; the team must decide how to complete that requirement before submission rather than asserting full compliance.

**Write a final reflection:** What did you personally implement? What evidence supports it? Which decisions need another owner's agreement? Which findings belong to the new pilot rather than the historical project? What remains before submission?


**Cell 24**

## Background references

- [GWOSC data](https://gwosc.org/data/)
- [PyCBC waveforms](https://pycbc.org/pycbc/latest/html/waveform.html)
- [SciPy signal processing](https://docs.scipy.org/doc/scipy/reference/signal.html)
- [sbi calibration guide](https://sbi.readthedocs.io/en/latest/how_to_guide/16_sbc.html)

Consult library documentation for concepts and conventions. Your notebook should contain your own implementation and interpretation.